# Tabelle der F-Verteilung – einfach erklärt

> Quelle: [numiqo.de](https://numiqo.de/tutorial/f-verteilung)

Die **F-Verteilung** ergibt sich aus dem Quotienten zweier Chi-Quadrat-Verteilungen,
dividiert durch ihre jeweiligen Freiheitsgrade:

$$F = \frac{\chi^2_1 / df_1}{\chi^2_2 / df_2}$$

**Zwei Freiheitsgrade:**
- $df_1$ = Zähler-Freiheitsgrade (between groups)
- $df_2$ = Nenner-Freiheitsgrade (within groups / Residuals)

**Anwendung:**

| Test | df₁ | df₂ |
|---|---|---|
| **Einfaktorielle ANOVA** | k − 1 | N − k |
| **Multiple Regression** | k | n − k − 1 |
| **Levene-Test** | k − 1 | N − k |

**Entscheidungsregel:**
$$F_{berechnet} > F_{krit}(df_1, df_2, \alpha) \Rightarrow H_0 \text{ verwerfen}$$

**F-Verteilung ist immer rechtsseitig** (F ≥ 0).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import f as f_dist

print("Bibliotheken erfolgreich geladen.")

## 1. F-Verteilung für verschiedene Freiheitsgrade

Die Form der F-Verteilung hängt von beiden Freiheitsgraden (df₁, df₂) ab.

In [ ]:
x = np.linspace(0.01, 8, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("F-Verteilung: Freiheitsgrade und kritische Werte", fontsize=12,
             fontweight="bold")

param_liste = [(1,10), (2,10), (3,10), (5,20), (10,30)]
farben = ["#E91E63", "#FF9800", "#4CAF50", "#2196F3", "#9C27B0"]

for (df1_v, df2_v), fc in zip(param_liste, farben):
    pdf_v = f_dist.pdf(x, df1_v, df2_v)
    axes[0].plot(x, pdf_v, color=fc, linewidth=2.5,
                 label="df1=" + str(df1_v) + ", df2=" + str(df2_v))

axes[0].set_title("Dichtefunktion f(F) je Freiheitsgrade(rechtsschiefe Verteilung)")
axes[0].set_xlabel("F")
axes[0].set_ylabel("Dichte")
axes[0].legend(fontsize=8)
axes[0].set_xlim(0, 7)
axes[0].set_ylim(0, 1.5)
axes[0].grid(alpha=0.3)

# F_krit Heatmap für alpha=0.05
df1_vals = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15]
df2_vals = [5, 10, 15, 20, 30, 60, 120]
mat = np.array([[f_dist.ppf(0.95, df1_v, df2_v) for df1_v in df1_vals]
                for df2_v in df2_vals])
im = axes[1].imshow(mat, cmap="RdYlGn_r", aspect="auto")
plt.colorbar(im, ax=axes[1], label="F_krit (alpha=0.05)")
axes[1].set_xticks(range(len(df1_vals)))
axes[1].set_yticks(range(len(df2_vals)))
axes[1].set_xticklabels(df1_vals, fontsize=8)
axes[1].set_yticklabels(df2_vals, fontsize=8)
axes[1].set_xlabel("df1 (Zähler)")
axes[1].set_ylabel("df2 (Nenner)")
axes[1].set_title("F_krit Heatmap (alpha=0.05)höher df2 -> kleineres F_krit")
for i in range(len(df2_vals)):
    for j in range(len(df1_vals)):
        axes[1].text(j, i, str(round(mat[i,j], 2)),
                     ha="center", va="center", fontsize=7,
                     color="white" if mat[i,j] > 5 else "black")

plt.tight_layout()
plt.show()

## 2. F-Tabelle α = 0,05 (numiqo)

**Zeilen** = df₂ (Nenner), **Spalten** = df₁ (Zähler)

In [ ]:
# F-Tabelle exakt wie numiqo (alpha=0.05)
df1_zeilen = list(range(1, 16))   # Zähler df1: 1-15
df2_zeilen = list(range(1, 16))   # Nenner  df2: 1-15

tabelle_05 = {}
for df2_v in df2_zeilen:
    row = {}
    for df1_v in df1_zeilen:
        f_krit_v = round(f_dist.ppf(0.95, df1_v, df2_v), 3)
        row[str(df1_v)] = f_krit_v
    tabelle_05[str(df2_v)] = row

df_tab05 = pd.DataFrame(tabelle_05).T
df_tab05.index.name = "df2\df1"
df_tab05.columns = [str(d) for d in df1_zeilen]

print("F-Tabelle (alpha=0.05, 1-alpha=0.95)")
print("Zeilen = df2 (Nenner), Spalten = df1 (Zähler)")
print()
print(df_tab05.to_string())
print()
print("Lesebeispiele (numiqo-Verifikation):")
print("df1=1, df2=1:   F_krit=" + str(round(f_dist.ppf(0.95, 1, 1), 3)) +
      "  (numiqo: 161.448)")
print("df1=2, df2=2:   F_krit=" + str(round(f_dist.ppf(0.95, 2, 2), 3)) +
      "  (numiqo: 19.000)")
print("df1=3, df2=10:  F_krit=" + str(round(f_dist.ppf(0.95, 3, 10), 3)) +
      "  (numiqo: 3.708)")
print("df1=1, df2=10:  F_krit=" + str(round(f_dist.ppf(0.95, 1, 10), 3)) +
      "  (numiqo: 4.965)")

In [ ]:
# F-Tabelle fuer alpha=0.01
tabelle_01 = {}
for df2_v in df2_zeilen:
    row = {}
    for df1_v in df1_zeilen:
        f_krit_v = round(f_dist.ppf(0.99, df1_v, df2_v), 3)
        row[str(df1_v)] = f_krit_v
    tabelle_01[str(df2_v)] = row

df_tab01 = pd.DataFrame(tabelle_01).T
df_tab01.index.name = "df2\df1"
df_tab01.columns = [str(d) for d in df1_zeilen]

print("F-Tabelle (alpha=0.01, 1-alpha=0.99")
print(df_tab01.to_string())
print()
print("Lesebeispiel:")
print("df1=1, df2=1:   F_krit=" + str(round(f_dist.ppf(0.99, 1, 1), 3)) +
      "  (numiqo: 4052.181)")
print("df1=3, df2=10:  F_krit=" + str(round(f_dist.ppf(0.99, 3, 10), 3)) +
      "  (numiqo: 6.552)")

In [ ]:
def p_aus_f(f_val, df1_v, df2_v):
    return 1 - f_dist.cdf(f_val, df1_v, df2_v)

def f_krit(df1_v, df2_v, alpha=0.05):
    return round(f_dist.ppf(1 - alpha, df1_v, df2_v), 4)

print("p-Wert aus F-Wert")
print()
print("F-Wert  df1   df2   p-Wert    signifikant?")
print("-" * 48)
for f_v, df1_v, df2_v in [(5.0, 2, 27), (3.9, 3, 20), (4.5, 1, 30),
                            (2.8, 4, 50), (10.0, 2, 10)]:
    p = round(p_aus_f(f_v, df1_v, df2_v), 4)
    sig = "JA" if p < 0.05 else "nein"
    print(str(round(f_v,1)).rjust(6) + "  " + str(df1_v).rjust(4) +
          "  " + str(df2_v).rjust(4) + "  " + str(p).ljust(8) + "  " + sig)

print()
print("ANOVA Beispiel: 3 Gruppen, n=10 je Gruppe")
k = 3
n_je = 10
N = k * n_je
df1_anova = k - 1
df2_anova = N - k
f_krit_anova = f_krit(df1_anova, df2_anova)
print("k=" + str(k) + " Gruppen, n=" + str(n_je) + " je Gruppe, N=" + str(N))
print("df1 = k-1 = " + str(df1_anova))
print("df2 = N-k = " + str(df2_anova))
print("F_krit (alpha=0.05) = " + str(f_krit_anova))
print()
print("-> F_berechnet > " + str(f_krit_anova) + " -> H0 verwerfen (signifikant)")

In [ ]:
df1_bsp = 2
df2_bsp = 27
f_bsp   = 5.0
f_krit_bsp = f_dist.ppf(0.95, df1_bsp, df2_bsp)
p_bsp = 1 - f_dist.cdf(f_bsp, df1_bsp, df2_bsp)

x_f = np.linspace(0.01, 10, 400)
pdf_f = f_dist.pdf(x_f, df1_bsp, df2_bsp)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("F-Verteilung: ANOVA-Hypothesentest", fontsize=12)

# 1. Kritischer Bereich
axes[0].plot(x_f, pdf_f, color="steelblue", linewidth=2.5)
axes[0].fill_between(x_f, pdf_f, where=(x_f >= f_krit_bsp),
                     color="tomato", alpha=0.6, label="Ablehnungsbereich (5%)")
axes[0].fill_between(x_f, pdf_f, where=(x_f < f_krit_bsp),
                     color="steelblue", alpha=0.15, label="Annahmebereich")
axes[0].axvline(f_krit_bsp, color="black", linestyle="--", linewidth=2,
                label="F_krit=" + str(round(f_krit_bsp, 3)))
axes[0].axvline(f_bsp, color="seagreen", linewidth=2.5,
                label="F_berechnet=" + str(f_bsp))
axes[0].set_title("df1=" + str(df1_bsp) + ", df2=" + str(df2_bsp) +
                  "F=" + str(f_bsp) + " > F_krit=" +
                  str(round(f_krit_bsp,3)) + " -> H0 verwerfen!")
axes[0].set_xlabel("F")
axes[0].set_ylabel("Dichte")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)
axes[0].set_xlim(0, 9)

# 2. p-Wert Fläche
axes[1].plot(x_f, pdf_f, color="steelblue", linewidth=2.5)
axes[1].fill_between(x_f, pdf_f, where=(x_f >= f_bsp),
                     color="tomato", alpha=0.6,
                     label="p=" + str(round(p_bsp,4)))
axes[1].axvline(f_bsp, color="black", linestyle="--", linewidth=2,
                label="F=" + str(f_bsp))
axes[1].set_title("p-Wert = " + str(round(p_bsp,4)) +
                  (" < 0.05 -> signifikant!" if p_bsp < 0.05 else ""))
axes[1].set_xlabel("F")
axes[1].set_ylabel("Dichte")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_xlim(0, 9)

plt.tight_layout()
plt.show()

## 3. Zusammenfassung

```
F-Verteilung Tabelle – Übersicht
│
├── DEFINITION
│   F = (chi2_1/df1) / (chi2_2/df2)
│   Zwei Freiheitsgrade: df1 (Zähler), df2 (Nenner)
│   Nur positive Werte, IMMER rechtsseitig
│
├── FREIHEITSGRADE
│   ANOVA:      df1 = k-1,  df2 = N-k
│   Regression: df1 = k,    df2 = n-k-1
│   Levene:     df1 = k-1,  df2 = N-k
│
├── TABELLE LESEN
│   Zeile  = df2 (Nenner-Freiheitsgrade)
│   Spalte = df1 (Zähler-Freiheitsgrade)
│   Zellwert = kritischer F-Wert
│   Separate Tabellen für alpha=0.05 und alpha=0.01
│
├── ENTSCHEIDUNGSREGEL
│   F_berechnet > F_krit -> H0 verwerfen (signifikant)
│   F_berechnet <= F_krit -> H0 beibehalten
│
├── BEISPIELE F_krit (alpha=0.05)
│   df1=1, df2=10:  F_krit = 4.965
│   df1=2, df2=10:  F_krit = 4.103
│   df1=3, df2=10:  F_krit = 3.708
│   df1=2, df2=27:  F_krit = 3.354
│
└── PYTHON
    from scipy.stats import f
    p_wert  = 1 - f.cdf(F_wert, df1, df2)
    F_krit  = f.ppf(1 - alpha, df1, df2)
```

---
Quelle: [numiqo.de/tutorial/f-verteilung](https://numiqo.de/tutorial/f-verteilung)